# 08 — Natural-Language Assistant
**RetailMind · Data Science Practicum 2 · Sai Teja Sunku**

A Q&A interface over the pipeline outputs. **Works offline by default**
(no API key) using keyword intent routing. With a Groq API key set, it
upgrades to LLM-powered answers grounded in the same pipeline JSON.

| Mode | When to use |
|---|---|
| `offline` | No API key needed. Deterministic answers with real numbers. |
| `groq`    | Richer prose. Needs `GROQ_API_KEY` (free at console.groq.com). |
| `auto`    | Picks groq if a key is available, else offline. |


In [1]:
# Common setup: make the project package importable from the notebooks/ folder
import sys, warnings, json
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px


## 1. Run the pipeline to populate context

In [2]:
from retailmind import RetailPipeline, ask

p = RetailPipeline.from_files('../train.csv', auxiliary_paths=['../store.csv'])
p.horizon = 14
p.max_entities_for_full_forecast = 5
p.run(cv_folds=2)
print('Pipeline ready.')

Pipeline ready.


## 2. Ask offline-mode questions

In [3]:
questions = [
    'Give me an overview of the dataset',
    'How accurate is the forecast?',
    'What drives sales the most?',
    'Show me the top anomalies',
    'How much should I order?',
    'How does promo affect sales?',
    "What's the day-of-week pattern?",
]
for q in questions:
    print('-' * 70)
    print('Q:', q)
    print(ask(p, q, mode='offline'))
    print()

----------------------------------------------------------------------
Q: Give me an overview of the dataset
**Dataset overview**
• 1,050,329 observations across 1115 entities
• Time span: 2013-01-01 → 2015-07-31 (942 days)
• Total sales: $5.87B
• Mean daily sales per entity: 5,592
• 80.4% of entity-days had non-zero sales

----------------------------------------------------------------------
Q: How accurate is the forecast?
**Forecast model performance (walk-forward CV)**
• RMSE:  841.8
• MAE:   552.3
• MAPE:  697.4%  (high when many zero-sales days exist)
• SMAPE: 31.4%  (more reliable than MAPE on retail data)

----------------------------------------------------------------------
Q: What drives sales the most?
**Top 5 sales drivers** (LightGBM gain importance, leakage-safe)
• `is_open` — ↑ increases sales
• `sales_lag_14` — ↑ increases sales
• `sales_lag_28` — ↑ increases sales
• `sales_lag_1` — ↑ increases sales
• `promo` — ↑ increases sales

-------------------------------------

## 3. Optional Groq mode
If you set `GROQ_API_KEY` in your environment (or pass it via `api_key=`),
the assistant will route through Groq for richer prose answers.

```python
answer = ask(p, "Write a 3-sentence summary for the store owner",
              mode='groq', api_key='gsk_...')
print(answer)
```


## Summary
- The assistant always answers — even without an API key
- Offline mode is keyword-routed and returns specific numbers from pipeline outputs
- Groq mode is a richer layer for prose summaries

**Next:** [09 — Universal Pipeline (showcase)](09_universal_pipeline.ipynb)
